In this notebook feature engineering and data cleaning are performed. The workflow is:
1. removing some features 
3. handling NaN values
2. adding feature to distinguish the single-hit event from multi-hit event



# 0 - General Ops

In [1]:
import pandas as pd
import numpy as np

import os
from pathlib import Path


In [2]:
project_dir = Path.cwd().parent
os.chdir(project_dir)
file_path = project_dir / "data" / "Background" / "Engineered_Datasets" 
file_name = "combinedbg.csv"

In [3]:
ds = pd.read_csv(file_path / file_name, header=0)
ds.shape

(324057, 54)

# 1 - Removing Features

In this section some features are removed. The reasons to remove a particular feature are:
- That feature is not available in real-case scenario: it is only available in a simulation context
- That feature is *redundant* with some other features

## Remove `primary_e0_kev` feture

The `primary_e0_kev` feature is **not available** in real-case scanario: it is the energy of the event at the start of the simulation (before hitting the detector).

In [4]:
if "primary_e0_kev" in ds.columns.to_list():
    ds = ds.drop(columns=["primary_e0_kev"])
    print(f"Column 'primary_e0_kev' removed, shape is now {ds.shape}.")
else:
    print(f"Column 'primary_e0_kev' not found, shape is {ds.shape}.")

Column 'primary_e0_kev' removed, shape is now (324057, 53).


## Remove *total hit*-like feature

The features `n_active`, `hits_sum`, `n_voxels` are redundant: all these features represents the number of activated sub-cubes (detector). Follow a demonstration:

In [5]:
if all(col in ds.columns.to_list() for col in ["n_voxels", "hits_sum","n_active"]):
    ds["hits_sum"] = ds["hits_sum"].astype(int)
    #print(f"Does the columns ['n_voxels', 'hits_sum','n_active'] have the same values? {ds["n_active"].equals(ds["n_voxels"]) and ds["n_active"].equals(ds["hits_sum"])}")
    print(f"n_voxels = hits_sum : {ds['n_voxels'].equals(ds['hits_sum'])}")
    print(f"n_voxels = n_active : {ds['n_voxels'].equals(ds['n_active'])}")
    print(f"hits_sum = n_active : {ds['hits_sum'].equals(ds['n_active'])}")

n_voxels = hits_sum : True
n_voxels = n_active : True
hits_sum = n_active : True


Totally drop `n_active` and `n_voxels`:

In [6]:
if all(col in ds.columns.to_list() for col in ["n_voxels", "hits_sum","n_active"]): 
    ds = ds.drop(columns=["n_voxels", "n_active"])
    print(f"Columns 'n_voxels' and 'n_active' removed, shape is now {ds.shape}.")
else: 
    print(f"Columns 'n_voxels' and 'n_active' not found, shape is {ds.shape}.")  

Columns 'n_voxels' and 'n_active' removed, shape is now (324057, 51).


## Remove `hits_mean_active` feature

The `hits_mean_active` feature contain just one unique value across avery sample:

In [7]:
if "hits_mean_active" in ds.columns.to_list():
    print(ds["hits_mean_active"].describe(include="all"))
    print(f"\nNumber of same value ({ds['hits_mean_active'].nunique()}) in 'hits_mean_active': {ds['hits_mean_active'].value_counts().max()} / {ds.shape[0]}")

count    324057.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: hits_mean_active, dtype: float64

Number of same value (1) in 'hits_mean_active': 324057 / 324057


Totally drop the column:

In [8]:
if "hits_mean_active" in ds.columns.to_list():
    ds = ds.drop(columns=["hits_mean_active"])
    print(f"Column 'hits_mean_active' removed, shape is now {ds.shape}.")

Column 'hits_mean_active' removed, shape is now (324057, 50).


# 2 - Handling NaN features

There are feature that contains only NaN values: 

In [9]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

nan_perc_df = pd.DataFrame(
    {"Column": nan_counts.index, "Total NaN": nan_counts.values, "Percentage": (nan_counts.values / ds.shape[0] * 100).round(2)}
).sort_values(by="Percentage", ascending=False).reset_index(drop=True)

nan_perc_df

,Column,Total NaN,Percentage
0,corr_E_hits,324057,100.00
1,f_occ,324057,100.00
2,D_f_r2,273418,84.37
3,E_skew,267836,82.65
4,E_kurt,267836,82.65
5,D_f,256365,79.11
6,E_std,230281,71.06
7,linearity,230281,71.06
8,planarity,230281,71.06
9,sphericity,230281,71.06


Drop those columns with 100% of NaN values:

In [10]:
columns_100perc_nan = nan_perc_df.iloc[:2]["Column"].tolist()  
ds = ds.drop(columns=columns_100perc_nan)
print(f"Columns {columns_100perc_nan} removed, shape is now {ds.shape}.")


Columns ['corr_E_hits', 'f_occ'] removed, shape is now (324057, 48).


## Hits/Shape and NaN correlation

Most of NaN are due to the low quantity of hits per event. Let's visualize the correlation between the NaN values and the number of hits (=1 or > 1)

In [11]:
one_hit = ds[ds["hits_sum"] == 1]
multi_hits = ds[ds["hits_sum"] > 1]

print(f"Number of events with one hit: {one_hit.shape[0]} ({one_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with multiple hits: {multi_hits.shape[0]} ({multi_hits.shape[0]/ds.shape[0]*100:.2f}%)")

bg = ds["particle_type"].unique()
one_hit_counts = [one_hit[one_hit["particle_type"] == b].shape[0] for b in bg]
one_hit_percentages = [one_hit[one_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
pd.DataFrame(
    {
        "Background": bg,
        "One Hit count": [f"{one_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        #"One Hit count": f"{one_hit_counts} / {ds[ds["particle_type"] == bg].shape[0]}",
        "One Hit percentage": np.round(one_hit_percentages, 2),
    }
)

Number of events with one hit: 230281 (71.06%)
Number of events with multiple hits: 93776 (28.94%)


,Background,One Hit count,One Hit percentage
0,SecondaryProtonsUpward,5902 / 11901,49.59
1,PrimaryPositrons,1 / 4,25.00
2,CosmicPhotons,137476 / 152860,89.94
3,PrimaryProtons,3575 / 18461,19.37
4,AlbedoPhotons,65667 / 86862,75.60
5,PrimaryElectrons,25 / 148,16.89
6,SecondaryElectrons,3134 / 8730,35.90
7,AtmosphericNeutrons,1364 / 2161,63.12
8,PrimaryAlphas,624 / 2781,22.44
9,SecondaryProtonsDownward,5486 / 11060,49.60


⚠️ Datasets must be balanced!

In [12]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

columns_with_nan = nan_counts.index

one_hit_nan_counts = one_hit[columns_with_nan].isna().sum()
multi_hits_nan_counts = multi_hits[columns_with_nan].isna().sum()
one_hit_nan_counts_percent = np.round(one_hit[columns_with_nan].isna().sum() / nan_counts[columns_with_nan] * 100, 2)
multi_hits_nan_counts_percent = np.round(multi_hits[columns_with_nan].isna().sum() / nan_counts[columns_with_nan] * 100, 2)

nan_report = pd.DataFrame({
    "NaN Count": nan_counts,
    "One Hit NaN Count": one_hit_nan_counts,
    "Multi Hits NaN Count": multi_hits_nan_counts,
    "One Hit NaN percentage": one_hit_nan_counts_percent,
    "Multi Hits NaN percentage": multi_hits_nan_counts_percent
})

nan_report.index.name = "Column"
nan_report.reset_index()

,Column,NaN Count,One Hit NaN Count,Multi Hits NaN Count,One Hit NaN percentage,Multi Hits NaN percentage
0,E_std,230281,230281,0,100.00,0.00
1,E_skew,267836,230281,37555,85.98,14.02
2,E_kurt,267836,230281,37555,85.98,14.02
3,linearity,230281,230281,0,100.00,0.00
4,planarity,230281,230281,0,100.00,0.00
5,sphericity,230281,230281,0,100.00,0.00
6,D_f,256365,230281,26084,89.83,10.17
7,D_f_r2,273418,230281,43137,84.22,15.78


The geometrical features and some energy features are obviously correlated with the number of hits: to define some of these features, at least 2 hits are required (e.g., standard deviation). In particular:
- E_std needs at least 2 hits,
- linearity, planarity and sphericity need at least 2,3+ hits
- E_skew (skewness) at lest 3 hits
- E_kurt 

Two different approach are possible: 
- setting those feature to a default value (e.g., 0) if the number of hits is equal to 1
- create 2 differents sub-datasets:
    - One-hit dataset which does not contain the NaN features
    - Multi-hit dataset and handle separately the remaining NaNs 

# 3 - Distinguish one/multi-hit events

In [13]:
one_hit = ds[ds["hits_sum"] == 1]
multi_hits = ds[ds["hits_sum"] > 1]

print(f"Number of events with one hit: {one_hit.shape[0]} ({one_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with multiple hits: {multi_hits.shape[0]} ({multi_hits.shape[0]/ds.shape[0]*100:.2f}%)")

bg = ds["particle_type"].unique()
one_hit_counts = [one_hit[one_hit["particle_type"] == b].shape[0] for b in bg]
one_hit_percentages = [one_hit[one_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
pd.DataFrame(
    {
        "Background": bg,
        "One Hit count": [f"{one_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        #"One Hit count": f"{one_hit_counts} / {ds[ds["particle_type"] == bg].shape[0]}",
        "One Hit percentage": np.round(one_hit_percentages, 2),
    }
).sort_values(by="One Hit percentage", ascending=False)

Number of events with one hit: 230281 (71.06%)
Number of events with multiple hits: 93776 (28.94%)


,Background,One Hit count,One Hit percentage
2,CosmicPhotons,137476 / 152860,89.94
4,AlbedoPhotons,65667 / 86862,75.60
7,AtmosphericNeutrons,1364 / 2161,63.12
9,SecondaryProtonsDownward,5486 / 11060,49.60
0,SecondaryProtonsUpward,5902 / 11901,49.59
6,SecondaryElectrons,3134 / 8730,35.90
1,PrimaryPositrons,1 / 4,25.00
10,SecondaryPositrons,7027 / 29089,24.16
8,PrimaryAlphas,624 / 2781,22.44
3,PrimaryProtons,3575 / 18461,19.37


Two possible approach:
- mantain one single dataset with a *flag*-like feature which distinguish between single or multi hits 
- divide two the dataset in two sub-datasets

NaN values must be handled!

In [14]:
ds["multi_hits"] = ds["hits_sum"] > 1
print(f"Number of events with multiple hits: {ds[ds["multi_hits"]].shape[0]} ({ds[ds["multi_hits"]].shape[0]/ds.shape[0]*100:.2f}%)")

Number of events with multiple hits: 93776 (28.94%)


In [15]:
one_hit = ds[ds["multi_hits"] == False]
multi_hits = ds[ds["multi_hits"] == True]

### One-hit Dataset

In [16]:
print(f"\"Raw\" One hit dataset shape: {one_hit.shape}")
print(f"\n\nNaN values in one hit dataset:")
oh_nan_val = one_hit.isna().sum()
oh_nan_val = oh_nan_val[oh_nan_val > 0]
print(oh_nan_val)

"Raw" One hit dataset shape: (230281, 49)


NaN values in one hit dataset:
E_std         230281
E_skew        230281
E_kurt        230281
linearity     230281
planarity     230281
sphericity    230281
D_f           230281
D_f_r2        230281
dtype: int64


Just simply drop this features

In [17]:
columns_to_drop = oh_nan_val.index.tolist()

if columns_to_drop:
    print(f"\nDropping columns with NaN values in one hit dataset: {columns_to_drop}")
    one_hit = one_hit.drop(columns=columns_to_drop)

    print(f"NaN values in one hit dataset after dropping NaN values: {one_hit.isna().sum().sum()}")

print(f'"Cleaned" One hit dataset shape: {one_hit.shape}')


Dropping columns with NaN values in one hit dataset: ['E_std', 'E_skew', 'E_kurt', 'linearity', 'planarity', 'sphericity', 'D_f', 'D_f_r2']
NaN values in one hit dataset after dropping NaN values: 0
"Cleaned" One hit dataset shape: (230281, 41)


### Multi-hit Dataset

In [18]:
print(f"\"Raw\" Multi hits dataset shape: {multi_hits.shape}")
print(f"\n\nNaN values in multi hits dataset:")
mh_nan_val = multi_hits.isna().sum()
mh_nan_val = mh_nan_val[mh_nan_val > 0]
print(mh_nan_val)

"Raw" Multi hits dataset shape: (93776, 49)


NaN values in multi hits dataset:
E_skew    37555
E_kurt    37555
D_f       26084
D_f_r2    43137
dtype: int64


To compute the skewness and the kurtosis coeff. at least 3 hits are needed. To avoid to create a third dataset, and since there are almost ~40% missing values, we can decide to totally drop these features.

Also R2 index is may be dropped. For the Fractal Dimension we decide to impute a default value 0, preserving the meaning of the feature: a too small to be computed fractal dimension may be set to 0.  


In [19]:
mh_cols_drop = ["E_skew", "E_kurt", "D_f_r2"]

if all(col in multi_hits.columns.to_list() for col in mh_cols_drop):
    print(f"\nDropping columns with NaN values in multi hits dataset: {mh_cols_drop}")
    multi_hits = multi_hits.drop(columns=mh_cols_drop)

    print(f"NaN values in multi hits dataset after dropping NaN values: {multi_hits.isna().sum().sum()}")

# impute 0 to D_f
if "D_f" in multi_hits.columns.to_list():
    multi_hits["D_f"] = multi_hits["D_f"].fillna(0)
    print(f"NaN values in 'D_f' column after imputation: {multi_hits['D_f'].isna().sum()}")

print(f'"Cleaned" Multi hits dataset shape: {multi_hits.shape}')


Dropping columns with NaN values in multi hits dataset: ['E_skew', 'E_kurt', 'D_f_r2']
NaN values in multi hits dataset after dropping NaN values: 26084
NaN values in 'D_f' column after imputation: 0
"Cleaned" Multi hits dataset shape: (93776, 46)


### Summary and saving 

In [20]:
print(f"Final shape of one hit dataset: {one_hit.shape}")
print(f"Final shape of multi hits dataset: {multi_hits.shape}")

Final shape of one hit dataset: (230281, 41)
Final shape of multi hits dataset: (93776, 46)


In [21]:
one_hit.to_csv(file_path / "one_hit_dataset.csv", index=False)
multi_hits.to_csv(file_path / "multi_hits_dataset.csv", index=False)